# 11 — Agent Trace Analytics Views

**Jackson and Jackson HR Digital**

The Hire Right agent's MLflow traces live in Unity Catalog as OpenTelemetry Delta tables
(`<prefix>_otel_spans`, `<prefix>_otel_annotations`, …) plus the MLflow-managed `<prefix>_trace_unified` view.
Those are deeply nested (`VARIANT` attributes, span arrays), so this notebook publishes four flat,
fully-commented views that Genie and AI/BI dashboards can query directly.

| View | Grain | Answers |
|---|---|---|
| `v_agent_traces` | 1 row per trace (agent request or eval) | volume, latency, tokens, interaction type, tools used, missing-score prompts |
| `v_agent_tool_calls` | 1 row per tool invocation | slowest / most-used tools, failures, candidates scored, email domains |
| `v_agent_llm_calls` | 1 row per LLM call | token usage, LLM latency, tool-calling behaviour |
| `v_agent_assessments` | 1 row per LLM-judge score | Safety / Relevance / JnJ Tone pass rates and rationales |

### Prerequisites
- `06a_setup_uc_tracing.ipynb` has run (trace tables exist and are populated)

**Next notebook:** `12_create_trace_genie_agent.ipynb`

## Setup

In [ ]:
dbutils.widgets.text("trace_catalog",      "bldemos",                      "UC Catalog (traces)")
dbutils.widgets.text("trace_schema",       "hrd_2030_agent_observability", "UC Schema (traces)")
dbutils.widgets.text("trace_table_prefix", "hrd_agent",                   "Trace Table Prefix")

In [ ]:
import os

trace_catalog      = dbutils.widgets.get("trace_catalog")      or os.getenv("TRACE_CATALOG", "bldemos")
trace_schema       = dbutils.widgets.get("trace_schema")       or os.getenv("TRACE_SCHEMA", "hrd_2030_agent_observability")
trace_table_prefix = dbutils.widgets.get("trace_table_prefix") or os.getenv("TRACE_TABLE_PREFIX", "hrd_agent")

target  = f"{trace_catalog}.{trace_schema}"
spans   = f"{target}.{trace_table_prefix}_otel_spans"
unified = f"{target}.{trace_table_prefix}_trace_unified"

print(f"Source spans   : {spans}")
print(f"Source unified : {unified}")
print(f"Views schema   : {target}")

## View Definitions

Each view is declared with a table comment and a comment on every column — Genie uses these
descriptions to map business language ("slowest tool", "judge pass rate") onto the right columns.

In [ ]:
VIEWS = []

# Tool functions carry @mlflow.trace(span_type="TOOL") AND are wrapped by LangChain autologging,
# so each tool call produces a TOOL span with a same-named TOOL child. Count only the outermost
# TOOL span as the tool call, otherwise call counts and durations are double-counted.
TOOL_FLAGGED_SPANS = f"""
  SELECT s.*,
         s.attributes:['mlflow.spanType']::string = 'TOOL'
           AND COALESCE(p.attributes:['mlflow.spanType']::string, '') <> 'TOOL' AS is_tool_call
  FROM {spans} s
  LEFT JOIN {spans} p
    ON p.trace_id = s.trace_id AND p.span_id = s.parent_span_id
"""

# ── v_agent_traces ────────────────────────────────────────────────────────────
VIEWS.append(dict(
    name="v_agent_traces",
    comment=(
        "One row per Hire Right agent trace (an end-to-end agent request or an LLM-judge evaluation record). "
        "Use for request volume, latency, token usage, interaction mix, tools used, and how often the agent "
        "had to ask the user for missing candidate scores."
    ),
    columns=[
        ("trace_id",                 "Unique MLflow trace ID. Join key to v_agent_tool_calls, v_agent_llm_calls, v_agent_assessments."),
        ("request_time",             "Timestamp (UTC) when the agent request started."),
        ("request_date",             "Calendar date (UTC) of request_time. Use for daily trends."),
        ("trace_source",             "'Agent request' = real agent invocation (app, playground, notebook, API). 'Evaluation' = offline LLM-judge evaluation record. Exclude 'Evaluation' for production metrics."),
        ("interaction_type",         "Business category of the request: 'Free-form chat', 'Offer email' (app sends a candidate offer via send_email), or 'LLM judge evaluation'."),
        ("response_mode",            "'Streaming' (predict_stream — app chat / playground), 'Synchronous' (predict — offer emails, notebook tests, API), or 'Offline' (evaluation)."),
        ("status",                   "Trace outcome: OK or ERROR."),
        ("latency_seconds",          "End-to-end agent latency in seconds."),
        ("user_message",             "The latest user message sent to the agent (the question being answered)."),
        ("final_response",           "The agent's final text answer returned to the user."),
        ("requested_missing_scores", "TRUE when the agent's answer asked the user to supply missing candidate scores (e.g. interview_score, culture_fit) before it could run the ML hiring prediction."),
        ("input_tokens",             "Total LLM prompt (input) tokens consumed across the trace."),
        ("output_tokens",            "Total LLM completion (output) tokens generated across the trace."),
        ("total_tokens",             "input_tokens + output_tokens for the trace. NULL for evaluation records."),
        ("span_count",               "Number of spans (steps) recorded in the trace."),
        ("llm_call_count",           "Number of LLM calls the agent made in the trace (agent loop iterations)."),
        ("tool_call_count",          "Number of tool invocations in the trace."),
        ("error_span_count",         "Number of spans that ended in error."),
        ("tools_used",               "Sorted distinct list of tool names invoked in the trace (e.g. predict_hiring_score, search_resumes, send_email, get_candidate)."),
        ("user_name",                "MLflow user tag recorded on the trace."),
        ("logged_model_id",          "MLflow LoggedModel ID (agent version) that produced the trace."),
    ],
    sql=f"""
WITH root AS (
  SELECT trace_id, name AS entry_point
  FROM {spans}
  WHERE COALESCE(parent_span_id, '') = ''
  QUALIFY ROW_NUMBER() OVER (PARTITION BY trace_id ORDER BY time DESC) = 1
),
span_stats AS (
  SELECT
    s.trace_id,
    COUNT(*)                                                              AS span_count,
    COUNT_IF(s.attributes:['mlflow.spanType']::string = 'CHAT_MODEL')     AS llm_call_count,
    COUNT_IF(s.is_tool_call)                                              AS tool_call_count,
    COUNT_IF(s.status.code = 'STATUS_CODE_ERROR')                         AS error_span_count,
    ARRAY_SORT(COLLECT_SET(CASE WHEN s.is_tool_call
                                THEN REGEXP_REPLACE(s.name, '^.*__', '') END)) AS tools_used
  FROM ({TOOL_FLAGGED_SPANS}) s
  GROUP BY s.trace_id
),
base AS (
  SELECT
    u.trace_id,
    u.request_time,
    u.state,
    u.execution_duration_ms,
    r.entry_point,
    u.tags,
    u.trace_metadata,
    (u.tags['mlflow.eval.requestId'] IS NOT NULL OR r.entry_point = 'root_span') AS is_eval,
    COALESCE(
      ELEMENT_AT(FILTER(FROM_JSON(u.request:request.input::string, 'array<struct<role:string,content:string>>'), m -> m.role = 'user'), -1).content,
      ELEMENT_AT(FILTER(FROM_JSON(u.request:messages::string,      'array<struct<role:string,content:string>>'), m -> m.role = 'user'), -1).content,
      u.request
    ) AS user_message,
    COALESCE(
      ELEMENT_AT(FILTER(FROM_JSON(u.response:output::string, 'array<struct<type:string,content:array<struct<text:string>>>>'), o -> o.type = 'message'), -1).content[0].text,
      u.response
    ) AS final_response,
    FROM_JSON(u.trace_metadata:['mlflow.trace.tokenUsage']::string,
              'struct<input_tokens:bigint,output_tokens:bigint,total_tokens:bigint>') AS token_usage
  FROM {unified} u
  LEFT JOIN root r USING (trace_id)
)
SELECT
  b.trace_id,
  b.request_time,
  CAST(b.request_time AS DATE),
  CASE WHEN b.is_eval THEN 'Evaluation' ELSE 'Agent request' END,
  CASE
    WHEN b.is_eval                                                  THEN 'LLM judge evaluation'
    WHEN b.user_message LIKE 'Call the send_email tool exactly once%' THEN 'Offer email'
    ELSE 'Free-form chat'
  END,
  CASE b.entry_point WHEN 'predict_stream' THEN 'Streaming'
                     WHEN 'predict'        THEN 'Synchronous'
                     ELSE 'Offline' END,
  b.state,
  ROUND(b.execution_duration_ms / 1000.0, 2),
  b.user_message,
  b.final_response,
  COALESCE(b.final_response ILIKE '%missing%score%' OR b.final_response ILIKE '%need%score%', FALSE),
  b.token_usage.input_tokens,
  b.token_usage.output_tokens,
  b.token_usage.total_tokens,
  s.span_count,
  s.llm_call_count,
  s.tool_call_count,
  s.error_span_count,
  s.tools_used,
  b.tags['mlflow.user'],
  b.trace_metadata:['mlflow.modelId']::string
FROM base b
LEFT JOIN span_stats s USING (trace_id)
""",
))

# ── v_agent_tool_calls ────────────────────────────────────────────────────────
VIEWS.append(dict(
    name="v_agent_tool_calls",
    comment=(
        "One row per tool invocation made by the Hire Right agent. Use for tool usage, slowest tools, "
        "tool failures, which candidates were scored by the ML model, and where offer emails were sent."
    ),
    columns=[
        ("trace_id",               "MLflow trace ID of the agent request that made this call. Join key to v_agent_traces."),
        ("span_id",                "Unique span ID of this tool call."),
        ("call_time",              "Timestamp (UTC) when the tool call started."),
        ("call_date",              "Calendar date (UTC) of call_time."),
        ("tool_name",              "Tool name: predict_hiring_score, search_resumes, send_email, get_candidate, get_top_candidates, get_candidates_by_job, get_pipeline_candidates, get_hire_analytics, query_genie."),
        ("tool_category",          "Backing service: 'ML model endpoint', 'Vector Search (resumes)', 'UC SQL function', 'Email (Mailgun)', 'Genie', 'Other'."),
        ("duration_seconds",       "Tool execution time in seconds."),
        ("status",                 "OK or ERROR."),
        ("error_message",          "Error message when status = ERROR, otherwise NULL."),
        ("tool_input",             "JSON arguments the LLM passed to the tool."),
        ("tool_output",            "Text/JSON the tool returned to the agent."),
        ("candidate_id",           "Candidate ID (e.g. C001) passed to the tool, when applicable (predict_hiring_score, get_candidate)."),
        ("email_recipient_domain", "For send_email calls: the recipient's email domain (e.g. databricks.com). NULL for other tools."),
    ],
    sql=f"""
SELECT
  s.trace_id,
  s.span_id,
  s.time,
  CAST(s.time AS DATE),
  REGEXP_REPLACE(s.name, '^.*__', ''),
  CASE
    WHEN s.name LIKE '%\\_\\_%'           THEN 'UC SQL function'
    WHEN s.name = 'search_resumes'        THEN 'Vector Search (resumes)'
    WHEN s.name = 'predict_hiring_score'  THEN 'ML model endpoint'
    WHEN s.name = 'send_email'            THEN 'Email (Mailgun)'
    WHEN s.name ILIKE '%genie%'           THEN 'Genie'
    ELSE 'Other'
  END,
  ROUND((s.end_time_unix_nano - s.start_time_unix_nano) / 1e9, 3),
  CASE WHEN s.status.code = 'STATUS_CODE_ERROR' THEN 'ERROR' ELSE 'OK' END,
  NULLIF(s.status.message, ''),
  s.attributes:['mlflow.spanInputs']::string,
  s.attributes:['mlflow.spanOutputs']::string,
  TRY_PARSE_JSON(s.attributes:['mlflow.spanInputs']::string):candidate_id::string,
  CASE WHEN s.name = 'send_email'
       THEN SUBSTRING_INDEX(TRY_PARSE_JSON(s.attributes:['mlflow.spanInputs']::string):to::string, '@', -1)
  END
FROM ({TOOL_FLAGGED_SPANS}) s
WHERE s.is_tool_call
""",
))

# ── v_agent_llm_calls ─────────────────────────────────────────────────────────
VIEWS.append(dict(
    name="v_agent_llm_calls",
    comment=(
        "One row per LLM (chat model) call made by the Hire Right agent. Use for token consumption, "
        "LLM latency, and how often the model decided to call tools versus answer directly."
    ),
    columns=[
        ("trace_id",             "MLflow trace ID of the agent request. Join key to v_agent_traces."),
        ("span_id",              "Unique span ID of this LLM call."),
        ("call_time",            "Timestamp (UTC) when the LLM call started."),
        ("call_date",            "Calendar date (UTC) of call_time."),
        ("model",                "LLM serving endpoint / model name (e.g. databricks-gpt-5-4)."),
        ("duration_seconds",     "LLM call latency in seconds."),
        ("status",               "OK or ERROR."),
        ("input_tokens",         "Prompt (input) tokens for this call."),
        ("output_tokens",        "Completion (output) tokens for this call."),
        ("total_tokens",         "input_tokens + output_tokens for this call."),
        ("finish_reason",        "'tool_calls' = the model asked to run tools; 'stop' = the model produced a final answer."),
        ("tool_calls_requested", "Number of tool calls the model requested in this response (0 for final answers)."),
    ],
    sql=f"""
SELECT
  s.trace_id,
  s.span_id,
  s.time,
  CAST(s.time AS DATE),
  COALESCE(s.attributes:['invocation_params']:model::string,
           s.attributes:['invocation_params']:endpoint::string),
  ROUND((s.end_time_unix_nano - s.start_time_unix_nano) / 1e9, 3),
  CASE WHEN s.status.code = 'STATUS_CODE_ERROR' THEN 'ERROR' ELSE 'OK' END,
  FROM_JSON(s.attributes:['mlflow.chat.tokenUsage']::string, 'struct<input_tokens:bigint,output_tokens:bigint,total_tokens:bigint>').input_tokens,
  FROM_JSON(s.attributes:['mlflow.chat.tokenUsage']::string, 'struct<input_tokens:bigint,output_tokens:bigint,total_tokens:bigint>').output_tokens,
  FROM_JSON(s.attributes:['mlflow.chat.tokenUsage']::string, 'struct<input_tokens:bigint,output_tokens:bigint,total_tokens:bigint>').total_tokens,
  TRY_PARSE_JSON(s.attributes:['mlflow.spanOutputs']::string):generations[0][0]:generation_info:finish_reason::string,
  COALESCE(SIZE(FROM_JSON(
    TRY_PARSE_JSON(s.attributes:['mlflow.spanOutputs']::string):generations[0][0]:message:additional_kwargs:tool_calls::string,
    'array<string>')), 0)
FROM {spans} s
WHERE s.attributes:['mlflow.spanType']::string = 'CHAT_MODEL'
""",
))

# ── v_agent_assessments ───────────────────────────────────────────────────────
VIEWS.append(dict(
    name="v_agent_assessments",
    comment=(
        "One row per quality assessment (LLM judge or human feedback) attached to a Hire Right trace. "
        "Judges: 'safety', 'relevance_to_query', 'JnJ Tone' (J&J HR communication standards). "
        "Use for judge pass rates and failure rationales."
    ),
    columns=[
        ("trace_id",     "MLflow trace ID that was assessed. Join key to v_agent_traces."),
        ("request_time", "Timestamp (UTC) of the assessed trace."),
        ("request_date", "Calendar date (UTC) of request_time."),
        ("judge_name",   "Assessment name: 'safety', 'relevance_to_query', or 'JnJ Tone'."),
        ("source_type",  "Who produced the assessment: LLM_JUDGE, HUMAN, or CODE."),
        ("judge_value",  "Raw judge verdict (e.g. 'yes' / 'no')."),
        ("passed",       "TRUE when the judge verdict is 'yes' (pass), FALSE when 'no' (fail)."),
        ("rationale",    "The judge's written explanation for its verdict."),
        ("assessed_at",  "Timestamp (UTC) when the assessment was recorded."),
    ],
    sql=f"""
SELECT
  u.trace_id,
  u.request_time,
  CAST(u.request_time AS DATE),
  a.assessment_name,
  a.source.source_type,
  a.feedback.value::string,
  CASE LOWER(a.feedback.value::string) WHEN 'yes' THEN TRUE WHEN 'no' THEN FALSE END,
  a.rationale,
  a.create_time
FROM {unified} u
LATERAL VIEW EXPLODE(u.assessments) t AS a
WHERE a.feedback IS NOT NULL
""",
))

print(f"{len(VIEWS)} views defined: {[v['name'] for v in VIEWS]}")

## Create the Views

In [ ]:
def _q(text):
    return text.replace("'", "\\'")

for v in VIEWS:
    cols = ",\n  ".join(f"`{c}` COMMENT '{_q(d)}'" for c, d in v["columns"])
    ddl = (
        f"CREATE OR REPLACE VIEW {target}.{v['name']} (\n  {cols}\n)\n"
        f"COMMENT '{_q(v['comment'])}'\n"
        f"AS {v['sql']}"
    )
    spark.sql(ddl)
    print(f"✓ {target}.{v['name']}  ({len(v['columns'])} columns)")

## Validate

Row counts plus a headline summary — the same numbers the Genie Agent will be asked about.

In [ ]:
for v in VIEWS:
    _n = spark.sql(f"SELECT COUNT(*) AS n FROM {target}.{v['name']}").collect()[0]["n"]
    print(f"{v['name']:<24} {_n:>8,} rows")

In [ ]:
display(spark.sql(f"""
    SELECT interaction_type,
           COUNT(*)                               AS traces,
           ROUND(AVG(latency_seconds), 1)         AS avg_latency_s,
           ROUND(PERCENTILE(latency_seconds, 0.95), 1) AS p95_latency_s,
           SUM(total_tokens)                      AS total_tokens,
           COUNT_IF(requested_missing_scores)     AS missing_score_prompts
    FROM {target}.v_agent_traces
    GROUP BY interaction_type
    ORDER BY traces DESC
"""))

In [ ]:
display(spark.sql(f"""
    SELECT tool_name, tool_category,
           COUNT(*)                          AS calls,
           ROUND(AVG(duration_seconds), 2)   AS avg_duration_s,
           COUNT_IF(status = 'ERROR')        AS errors
    FROM {target}.v_agent_tool_calls
    GROUP BY tool_name, tool_category
    ORDER BY avg_duration_s DESC
"""))

## Summary

Four Genie-ready views now sit next to the raw trace tables in the trace schema. Because they are
views over the UC trace tables, they are always current — new traces from the serving endpoint
appear immediately with no refresh job.